# Transcription-factor activity along thymic ageing, scored on differential-expression statistics — CollecTRI regulons

**Question.** Which transcription factors (TFs) change activity with age in each thymic cell type,
and in particular do the steroid-hormone receptors (AR, NR3C1, PGR, ESR1, ESR2) gain activity in the
thymic stroma?

**Approach.** For every cell type, the DESeq2 Wald statistics of its age contrasts (paediatric vs
infant, adult vs infant) are scored against each CollecTRI TF regulon with decoupler's univariate
linear model, and calibrated against a permutation null that matches target-gene expression level.
Each TF is then reported as a **percentile among all TFs** in that cell type. The notebook is
self-contained (all functions are defined inline) and produces:

1. the per-cell-type × TF score table (Supplementary Table: raw score, permutation null, calibrated
   `z`, FDR, percentile);
2. violin plots placing the five steroid receptors within the distribution of all TFs, in the stroma
   and across all cell types;
3. the 20 regulons most consistently up- and down-regulated across the stromal populations;
4. a stress-response control (are the steroid signals just the generic stress programme?), which is
   what justifies carrying AR / NR3C1 / PGR forward and not ESR1 / ESR2;
5. the target genes that drive the AR, NR3C1 and PGR signals in the stroma.

Inputs: the DESeq2 result tables (one CSV per contrast, `cell_type, gene, baseMean, stat, ...`), the
curated cell-type hierarchy (`data/thyAgeing_celltype_levels.csv`), CollecTRI (downloaded via
decoupler on first use and cached in `data/`), and the MSigDB Hallmark stress gene sets
(`data/msigdb_hallmark_stress_2025.1.Hs.gmt`). Runtime ≈ 5 min on 4 cores (1,000 permutations).


In [1]:
import os, re, sys, time, warnings
import numpy as np
import pandas as pd
import decoupler as dc
from scipy.stats import norm, spearmanr, percentileofscore
from statsmodels.stats.multitest import multipletests
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, to_rgb
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
print(f'python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | '
      f'decoupler {dc.__version__} | matplotlib {matplotlib.__version__} | seaborn {sns.__version__}')


python 3.12.0 | numpy 2.2.4 | pandas 2.2.3 | decoupler 2.1.1 | matplotlib 3.10.1 | seaborn 0.13.2


In [2]:
# ------------------------------------------------------------------ paths (edit for your setup)
DEA_DIR = '/nfs/team205/vk8/projects/thymus_ageing_atlas/General_analysis/data/analyses/dea'  # DESeq2 tables
HERE    = os.environ.get('NB_OUT_DIR', os.getcwd())          # this folder: data/, results/, figures/
DATA_DIR, RESULTS, FIGURES = f'{HERE}/data', f'{HERE}/results', f'{HERE}/figures'
for d in (RESULTS, FIGURES):
    os.makedirs(d, exist_ok=True)

# ------------------------------------------------------------------ analysis settings
LEVEL       = 'taa_l2_v2'                              # annotation level of the DE tables
CONTRASTS   = ['paed_vs_infant', 'adult_vs_infant']    # panel order in every figure
NET_NAME    = 'CollecTRI'
PREFIX      = f'thyAgeing_{NET_NAME}_ageDE_{LEVEL}'
STEROID_TFS = ['AR', 'NR3C1', 'PGR', 'ESR1', 'ESR2']
STROMA_L1   = ['TEC', 'Fb', 'EC', 'Mural']             # broad lineages (taa_l1) that define "stroma"

# scoring parameters
TMIN     = 5                                            # min. measured targets for a regulon to be scored
STAT_CAP = 20.0                                         # |Wald statistic| cap
N_BINS   = 20                                           # baseMean quantile bins of the permutation null
N_PERM   = int(os.environ.get('NB_N_PERM', 1000))       # permutations per cell type x contrast
SEED     = 0
FORCE_RESCORE = os.environ.get('NB_FORCE_RESCORE', '0') == '1'   # recompute even if the table exists

SCORES_CSV = f'{RESULTS}/{PREFIX}_scores_perCellType.csv'        # the Supplementary Table
print(f'{NET_NAME} | {LEVEL} | outputs in {HERE}')


CollecTRI | taa_l2_v2 | outputs in /nfs/team205/vk8/projects/thymus_ageing_atlas/General_analysis/notebooks/hormone2cell/_builders/../final_versions/github


In [3]:
# ------------------------------------------------------------------ figure style
import logging
logging.getLogger('fontTools').setLevel(logging.WARNING)   # PDF font subsetting is chatty at INFO
logging.getLogger('matplotlib').setLevel(logging.WARNING)
try:                                                    # Arial if available (Nature house font)
    font_manager.fontManager.addfont('/nfs/team205/ny1/ThymusSpatialAtlas/software/Arial.ttf')
except Exception:
    pass
rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'pdf.fonttype': 42, 'ps.fonttype': 42,             # text stays editable in Illustrator
    'font.size': 7, 'axes.titlesize': 8, 'axes.labelsize': 7,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7, 'figure.titlesize': 8,
    'axes.linewidth': 0.5, 'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
    'xtick.major.size': 2, 'ytick.major.size': 2, 'xtick.major.pad': 1.5, 'ytick.major.pad': 1.5,
    'savefig.dpi': 300, 'savefig.bbox': 'standard',    # 'standard': the PDF keeps the size in mm below
    'savefig.transparent': False, 'figure.dpi': 110,
})
MM = 1 / 25.4                                            # inches per mm
NATURE_W = {'single': 89.0, 'two-thirds': 120.0, 'double': 183.0}   # Nature figure widths, mm
MISSING = '#e3e0e0'                                      # cell type not tested in a contrast
DIVERGING = sns.color_palette('vlag', as_cmap=True).copy()
DIVERGING.set_bad(MISSING)

# One colour + marker per steroid receptor (identical across all figures of the manuscript).
RECEPTOR_STYLE = {'AR':    ('#d95f02', 'o'),
                  'NR3C1': ('#1b9e77', '*'),
                  'PGR':   ('#7570b3', 'D'),
                  'ESR1':  ('#b2182b', 's'),
                  'ESR2':  ('#56B4E9', '^')}


def mm_fig(width_mm, height_mm):
    """A figure of exactly this size in millimetres; constrained layout keeps labels inside it."""
    return plt.figure(figsize=(width_mm * MM, height_mm * MM), layout='constrained')


def save_fig(fig, name):
    """Save `FIGURES/<name>.pdf` at the figure's size in mm, and report it."""
    path = f'{FIGURES}/{name}.pdf'
    fig.savefig(path)
    w, h = np.asarray(fig.get_size_inches()) * 25.4
    print(f'saved {os.path.basename(path)}  ({w:.0f} x {h:.0f} mm)')
    return path


In [4]:
# ------------------------------------------------------------------ cell-type order and lineage
# `thyAgeing_celltype_levels.csv` is the curated annotation hierarchy of the atlas (one row per
# finest label; columns taa_l1, taa_l2_v2, taa_l3, ... in the atlas' display order). It fixes the
# order of cell types in every figure and defines the lineages: "stroma" = TEC + Fb + EC + Mural.
CT_LEVELS = pd.read_csv(f'{DATA_DIR}/thyAgeing_celltype_levels.csv', dtype=str)
assert LEVEL in CT_LEVELS.columns, f'{LEVEL} is not a column of thyAgeing_celltype_levels.csv'
LINEAGE = (CT_LEVELS[[LEVEL, 'taa_l1']].dropna().drop_duplicates(LEVEL)
           .set_index(LEVEL)['taa_l1'].to_dict())          # cell type -> broad lineage (taa_l1)


def ct_order(lineages=None, present=None, level=LEVEL):
    """Cell types of `level` in atlas order, restricted to taa_l1 `lineages` and/or to `present`."""
    d = CT_LEVELS if lineages is None else CT_LEVELS[CT_LEVELS['taa_l1'].isin(lineages)]
    cts = list(dict.fromkeys(d[level].dropna()))
    return cts if present is None else [c for c in cts if c in set(present)]


# lineage colours: one distinct hue per broad lineage (stromal lineages first; the immune lineages
# use hues that do not collide with them), so points of different lineages never share a colour
_lin_all = list(dict.fromkeys(CT_LEVELS['taa_l1'].dropna()))
LINEAGE_COLORS = {'TEC': '#0072B2', 'Fb': '#E69F00', 'EC': '#009E73', 'Mural': '#CC79A7',
                  'T': '#D62728', 'NK': '#9467BD', 'B': '#8C564B', 'Mono': '#7F7F7F',
                  'Mac': '#BCBD22', 'DC': '#17BECF', 'Myeloid_dev': '#000000',
                  'Adipocyte': '#FF9896', 'Neutrophil': '#C5B0D5', 'Mast': '#C49C94'}
_extra = sns.color_palette('Set2', n_colors=8)                    # fallback for any other lineage
for i, l in enumerate([l for l in _lin_all if l not in LINEAGE_COLORS]):
    LINEAGE_COLORS[l] = matplotlib.colors.to_hex(_extra[i % len(_extra)])
print(f'{len(_lin_all)} lineages (taa_l1): {", ".join(_lin_all)}')
print(f'{CT_LEVELS[LEVEL].nunique()} cell types at {LEVEL}; stroma = {ct_order(lineages=STROMA_L1)}')


14 lineages (taa_l1): T, NK, TEC, Fb, Adipocyte, EC, Mural, B, Myeloid_dev, Neutrophil, Mono, Mac, DC, Mast
30 cell types at taa_l2_v2; stroma = ['cTEC', 'mcTEC', 'mTEC', 'TEC-EMT', 'Fb', 'EC-blood', 'EC-lymphatic', 'SMC', 'Pericyte']


## 1. Input: per-cell-type differential expression statistics

For every cell type, DESeq2 was run on donor-level pseudobulk profiles with age group as the
covariate and *infant* as the reference; the two contrasts used here are **paediatric vs infant**
and **adult vs infant** (see the differential-expression methods). The quantity scored below is the
**Wald statistic** (`stat`) of every gene DESeq2 tested in that cell type, together with the gene's
mean normalised expression (`baseMean`), which the permutation null is matched on.

Why the Wald statistic rather than expression: a footprint score computed on expression tracks the
*abundance* of a regulon's targets (highly expressed regulons score high whether or not anything
changes). A Wald statistic is a contrast centred on zero, so abundance cannot masquerade as
activity. The permutation null below removes the residual abundance dependence of the score.

The statistic is capped at ±20 so that a degenerate DESeq2 fit (`lfcSE → 0`) cannot set the scale of
its whole cell type.


In [5]:
DE_COLS = ['cell_type', 'gene_name', 'baseMean', 'stat']


def read_de_table(path):
    """One contrast: flat `cell_type, gene_name, baseMean, stat`. The gene column is `gene_name` or `gene`."""
    head = pd.read_csv(path, nrows=0).columns
    gene_col = 'gene_name' if 'gene_name' in head else 'gene'
    return (pd.read_csv(path, usecols=['cell_type', gene_col, 'baseMean', 'stat'])
              .rename(columns={gene_col: 'gene_name'})[DE_COLS])


def load_de(level, contrasts, dea_dir=DEA_DIR, stat_cap=STAT_CAP):
    """Both contrasts in one long table, Wald statistic capped at +/- stat_cap."""
    de = pd.concat([read_de_table(f'{dea_dir}/thyAgeing_dea_{level}_{c}_ageEffect.csv').assign(contrast=c)
                    for c in contrasts], ignore_index=True)
    assert np.isfinite(de['stat']).all(), 'non-finite Wald statistics in the DE tables'
    hit = de.loc[de['stat'].abs() > stat_cap, ['cell_type', 'contrast', 'gene_name', 'baseMean', 'stat']]
    print(f'{len(hit)} of {len(de):,} Wald statistics capped at +/-{stat_cap:g}')
    if len(hit):
        display(hit)
    de['stat'] = de['stat'].clip(-stat_cap, stat_cap)
    return de


de = load_de(LEVEL, CONTRASTS)

ct_by_contrast = {c: sorted(de.loc[de['contrast'] == c, 'cell_type'].unique()) for c in CONTRASTS}
shared_cts = sorted(set.intersection(*map(set, ct_by_contrast.values())))
for c, v in ct_by_contrast.items():
    extra = sorted(set(v) - set(shared_cts))
    print(f'{c}: {len(v)} cell types' + (f'  (only in this contrast: {", ".join(extra)})' if extra else ''))
n_by_ct = de.groupby('cell_type')['gene_name'].nunique()
print(f'{len(n_by_ct)} cell types in total, tested in both contrasts: {len(shared_cts)}')
print(f'genes tested per cell type: {n_by_ct.min():,} - {n_by_ct.max():,} (median {int(n_by_ct.median()):,})')
display(n_by_ct.reindex(ct_order(present=n_by_ct.index)).to_frame('genes tested').T)


2 of 870,930 Wald statistics capped at +/-20


,cell_type,contrast,gene_name,baseMean,stat
629836,TEC-EMT,adult_vs_infant,SCGB2A2,6.103285,3.361936e+07
632220,TEC-EMT,adult_vs_infant,IGHV2-5,6.669867,5.170339e+04


paed_vs_infant: 23 cell types
adult_vs_infant: 26 cell types  (only in this contrast: EC-lymphatic, Mono, Myeloid_dev)


26 cell types in total, tested in both contrasts: 23
genes tested per cell type: 8,722 - 27,510 (median 17,585)


cell_type,T_DN(early),T_DN,T_DP,T_αβT(entry),T_mature,T_recirc,T_innate,NK,cTEC,mcTEC,mTEC,TEC-EMT,Fb,EC-blood,EC-lymphatic,SMC,Pericyte,B_cell,B_plasma,Myeloid_dev,Mono,Mac,DC1,DC2,aDC,pDC
genes tested,15279,20652,23382,18467,20913,20087,18483,13505,20012,21699,27510,17274,23646,22126,13490,17897,15498,20005,15027,8722,11482,14048,13352,12979,13273,13504


## 2. Network: CollecTRI TF regulons

CollecTRI (Müller-Dott et al., 2023) is a curated collection of signed TF–target interactions
(weight +1 activation, −1 repression), obtained through `decoupler` (Zenodo record 8192729) and
cached in `data/` so that the exact version used is shipped with the analysis.


In [6]:
NET_CSV = f'{DATA_DIR}/CollecTRI_regulons_human.csv'
if os.path.exists(NET_CSV):
    net = pd.read_csv(NET_CSV)
    print('CollecTRI read from', NET_CSV)
else:
    net = dc.op.collectri(organism='human')[['source', 'target', 'weight']]
    net.to_csv(NET_CSV, index=False)
    print('CollecTRI downloaded through decoupler and cached to', NET_CSV)
net = net.drop_duplicates(['source', 'target']).reset_index(drop=True)
net_sources = set(net['source'])
n_tg = net.groupby('source').size()
print(f'{len(net_sources):,} TF regulons | {len(net):,} TF-target edges | {net["target"].nunique():,} target genes | '
      f'targets per regulon: median {int(n_tg.median())}, range {n_tg.min()}-{n_tg.max()} | '
      f'weights {sorted(net["weight"].unique())}')
measured = set(de['gene_name'])
cov = net.assign(measured=net['target'].isin(measured)).groupby('source')['measured'].agg(['size', 'sum'])
display(cov.reindex(STEROID_TFS).rename(columns={'size': 'targets in CollecTRI', 'sum': 'targets measured in the DE'}).T)


CollecTRI read from /nfs/team205/vk8/projects/thymus_ageing_atlas/General_analysis/notebooks/hormone2cell/_builders/../final_versions/github/data/CollecTRI_regulons_human.csv
1,185 TF regulons | 42,990 TF-target edges | 6,675 target genes | targets per regulon: median 9, range 1-1304 | weights [np.float64(-1.0), np.float64(1.0)]


source,AR,NR3C1,PGR,ESR1,ESR2
targets in CollecTRI,514,292,133,634,208
targets measured in the DE,506,288,130,622,206


## 3. Scoring: univariate linear model on the Wald statistics, calibrated against an abundance-matched permutation null

Every cell type is scored **on its own tested genes** (nothing is discarded to build a gene universe
shared across cell types). For one cell type × contrast:

1. **Adjacency.** A genes × regulons weight matrix over the tested genes (0 for non-targets),
   restricted to regulons with at least `TMIN` measured targets and column-standardised — exactly what
   `decoupler`'s `ulm` builds internally.
2. **Score.** The univariate linear model of decoupler: for each regulon, the Pearson correlation `r`
   between the vector of Wald statistics and the regulon's weight vector, reported as the t-value
   `score = r · sqrt(df / (1 − r²))`, `df = n_genes − 2`. Because the adjacency is fixed for a cell
   type, the score of any vector is a single matrix product, which is what makes 1,000 permutations
   affordable. The next cell checks that this reproduces `dc.mt.ulm` exactly.
3. **Null.** Genes are placed in `N_BINS` quantile bins of `baseMean`, and the Wald statistics are
   permuted *within* bins `N_PERM` times. Each permutation is scored; `null_mean` / `null_sd` are the
   mean and SD of a regulon's score across permutations, i.e. what a random gene set of the same size
   **and the same expression-level profile** scores in that cell type.
4. **Calibrated statistic.** `z = (score − null_mean) / null_sd`, `p_z = 2·Φ(−|z|)`, plus the empirical
   `p_emp`. `padj` is Benjamini–Hochberg across cell types within each regulon × contrast.
5. **Percentile.** `pct_z` is the percentile rank of a regulon's `z` among all regulons of the network
   scoreable in that cell type × contrast (100 = the regulon with the highest `z`). Because the ULM
   t-value grows with `sqrt(n_genes)` and the permutation does not remove that scaling, `z` is
   comparable *within* a cell type but not between cell types with different gene counts; the
   percentile (or `r`, which is scale-free) is what to compare across populations.


In [7]:
EPS = 2.2e-16                     # decoupler's guard in its t-value


def build_adjacency(universe, net, tmin=TMIN):
    """Genes x regulons weight matrix over `universe` (sorted gene symbols), column-standardised.
    Regulons with fewer than `tmin` targets in the universe are dropped (decoupler's `tmin`)."""
    gi = pd.Series(np.arange(len(universe)), index=universe)
    sub = net[net['target'].isin(gi.index)]
    n_tg = sub.groupby('source')['target'].size()
    srcs = sorted(n_tg.index[n_tg >= tmin])
    sub = sub[sub['source'].isin(set(srcs))]
    si = pd.Series(np.arange(len(srcs)), index=srcs)
    A = np.zeros((len(universe), len(srcs)), dtype=np.float64)
    A[gi[sub['target']].to_numpy(), si[sub['source']].to_numpy()] = sub['weight'].to_numpy()
    A_std = (A - A.mean(axis=0)) / A.std(axis=0, ddof=1)
    return srcs, A_std, n_tg.reindex(srcs).to_numpy()


def ulm(Y, A_std):
    """decoupler's univariate linear model written out: every row of Y (a vector over the universe)
    against every column of A_std. Returns (t, r) with t = r*sqrt(df/(1-r^2)), df = n_genes - 2."""
    Y = np.asarray(Y, dtype=np.float64)          # float64: float32 accumulation drifts ~2e-4
    n = Y.shape[1]
    Y_std = (Y - Y.mean(axis=1, keepdims=True)) / Y.std(axis=1, ddof=1, keepdims=True)
    r = (Y_std @ A_std) / (n - 1)
    return r * np.sqrt((n - 2) / ((1.0 - r + EPS) * (1.0 + r + EPS))), r


def abundance_bins(basemean, n_bins=N_BINS):
    """Quantile bin of baseMean per gene (ranked first so ties split deterministically)."""
    return pd.qcut(pd.Series(basemean).rank(method='first'), n_bins, labels=False).to_numpy()


def permutation_null(y, basemean, A_std, n_perm=N_PERM, seed=SEED, batch=200):
    """n_perm x n_regulons scores of `y` permuted WITHIN abundance bins (a regulon keeps its
    targets; the statistics are shuffled among genes of similar expression level)."""
    bins = abundance_bins(basemean)
    order = np.argsort(bins, kind='stable')
    g = bins[order].astype(np.float64)
    y_sorted = np.asarray(y, dtype=np.float64)[order]
    rng = np.random.default_rng(seed)
    out = np.empty((n_perm, A_std.shape[1]), dtype=np.float32)
    for start in range(0, n_perm, batch):
        k = min(batch, n_perm - start)
        keys = g[None, :] + rng.random((k, g.size))      # sorting keeps each bin's block intact
        out[start:start + k] = ulm(y_sorted[np.argsort(keys, axis=1, kind='stable')], A_std)[0]
    return out


def score_cell_type(d_ct, net, contrasts, n_perm=N_PERM, drop_genes=None):
    """All regulons in one cell type (the rows of `de` for it), on its own tested genes, per contrast."""
    ct = d_ct['cell_type'].iloc[0]
    universe = np.sort(d_ct['gene_name'].unique())
    if drop_genes is not None:                           # remove genes from the universe entirely
        universe = universe[~np.isin(universe, np.array(sorted(drop_genes)))]
    srcs, A_std, n_tg = build_adjacency(universe, net)
    rows = []
    for contrast in contrasts:
        d = d_ct[d_ct['contrast'] == contrast]
        if d.empty:
            continue
        d = d.set_index('gene_name').reindex(universe)
        y = d['stat'].to_numpy()
        t_obs, r_obs = ulm(y[None, :], A_std)
        null = permutation_null(y, d['baseMean'].to_numpy(), A_std, n_perm=n_perm)
        mu, sd = null.mean(axis=0), null.std(axis=0, ddof=1)
        z = (t_obs[0] - mu) / np.where(sd > 0, sd, np.nan)
        p_emp = (1 + (np.abs(null - mu) >= np.abs(t_obs[0] - mu)).sum(axis=0)) / (n_perm + 1)
        rows.append(pd.DataFrame({'cell_type': ct, 'contrast': contrast, 'source': srcs,
                                  'n_genes': len(universe), 'n_targets': n_tg,
                                  'score': t_obs[0], 'r': r_obs[0], 'null_mean': mu, 'null_sd': sd,
                                  'z': z, 'p_z': 2 * norm.sf(np.abs(z)), 'p_emp': p_emp}))
    return pd.concat(rows, ignore_index=True)


def score_all(de, net, contrasts=CONTRASTS, cell_types=None, n_perm=N_PERM, drop_genes=None):
    """Every cell type x contrast; `padj` = Benjamini-Hochberg across cell types within regulon x contrast."""
    cts = sorted(de['cell_type'].unique()) if cell_types is None else list(cell_types)
    out, t0 = [], time.time()
    for i, ct in enumerate(cts, 1):
        r = score_cell_type(de[de['cell_type'] == ct], net, contrasts, n_perm, drop_genes)
        out.append(r)
        print(f'[{i}/{len(cts)}] {ct}: {r["n_genes"].iloc[0]:,} genes, {r["source"].nunique()} regulons '
              f'({time.time() - t0:.0f} s)', flush=True)
    res = pd.concat(out, ignore_index=True)
    res['padj'] = (res.groupby(['source', 'contrast'], observed=True)['p_z']
                      .transform(lambda p: multipletests(p, method='fdr_bh')[1]))
    return res


def add_percentiles(res, pool, by=('cell_type', 'contrast'), suffix=''):
    """`pct_z` / `pct_score`: percentile rank (100 = top) of z / raw score among the regulons of
    `pool` within each cell type x contrast; `n_pool` = number of regulons ranked against."""
    m = res['source'].isin(pool)
    g = res[m].groupby(list(by), observed=True)
    res.loc[m, 'pct_z' + suffix] = g['z'].rank(pct=True) * 100
    res.loc[m, 'pct_score' + suffix] = g['score'].rank(pct=True) * 100
    res.loc[m, 'n_pool' + suffix] = g['z'].transform('size')
    return res


print(f'TMIN={TMIN} | |stat| cap {STAT_CAP:g} | {N_BINS} abundance bins | {N_PERM} permutations | seed {SEED}')


TMIN=5 | |stat| cap 20 | 20 abundance bins | 1000 permutations | seed 0


In [8]:
def check_against_decoupler(de, net, cell_types, contrast='adult_vs_infant'):
    """The matrix-product ULM must reproduce `dc.mt.ulm` on the observed statistics."""
    for ct in cell_types:
        d = de[(de['cell_type'] == ct) & (de['contrast'] == contrast)].sort_values('gene_name')
        universe = d['gene_name'].to_numpy()
        srcs, A_std, _ = build_adjacency(universe, net)
        mine = ulm(d['stat'].to_numpy()[None, :], A_std)[0][0]
        M = pd.DataFrame([d['stat'].to_numpy()], index=[ct], columns=universe)
        theirs = dc.mt.ulm(data=M, net=net, tmin=TMIN, empty=False)[0].reindex(columns=srcs)
        np.testing.assert_allclose(mine, theirs.to_numpy()[0], rtol=1e-6, atol=1e-6)
        print(f'{ct}: reproduces dc.mt.ulm over {len(srcs)} regulons and {len(universe):,} genes')


# checked at both ends of the gene-count range, i.e. the smallest and the largest universe
check_against_decoupler(de, net, [n_by_ct.idxmin(), n_by_ct.idxmax()])


Myeloid_dev: reproduces dc.mt.ulm over 514 regulons and 8,722 genes


mTEC: reproduces dc.mt.ulm over 766 regulons and 27,510 genes


## 4. Score every cell type → Supplementary Table

One row per cell type × contrast × TF. Columns:

| column | meaning |
| --- | --- |
| `cell_type`, `contrast` | population and age contrast (infant is the reference) |
| `source` | TF regulon (CollecTRI) |
| `n_genes` | genes DESeq2 tested in this cell type (= the gene universe the regulon was scored on) |
| `n_targets` | regulon targets among those genes |
| `score` | raw ULM t-value of the Wald statistics against the regulon (= `decoupler.mt.ulm`) |
| `r` | the underlying Pearson correlation (scale-free effect size; comparable across cell types) |
| `null_mean`, `null_sd` | mean / SD of `score` over the abundance-matched permutations |
| `z` | `(score − null_mean) / null_sd`: calibrated activity change; > 0 = activity up with age |
| `p_z`, `p_emp` | two-sided normal and empirical permutation p-values of `z` |
| `padj` | Benjamini–Hochberg FDR of `p_z` across cell types, within regulon × contrast |
| `pct_z` | percentile of `z` among all TF regulons scoreable in that cell type × contrast (100 = top) |
| `pct_score` | the same percentile on the raw `score`, for reference |
| `n_pool` | number of TF regulons the percentile is taken over |

`pct_z` is the number to quote when comparing a TF across cell types; `z` compares TFs within a cell
type; `r` is the effect size. The scoring run takes about 3 min on 4 cores; the table is cached and reloaded
on later runs (set `FORCE_RESCORE = True` to recompute).


In [9]:
TABLE_COLS = ['cell_type', 'contrast', 'source', 'n_genes', 'n_targets', 'score', 'r', 'null_mean', 'null_sd',
              'z', 'p_z', 'p_emp', 'padj', 'pct_z', 'pct_score', 'n_pool']
if FORCE_RESCORE or not os.path.exists(SCORES_CSV):
    res = score_all(de, net)
    res = add_percentiles(res, pool=net_sources)[TABLE_COLS]
    res.to_csv(SCORES_CSV, index=False)
    print('written', SCORES_CSV)
else:
    res = pd.read_csv(SCORES_CSV)
    print('loaded cached scores from', SCORES_CSV)
print(f'{len(res):,} rows | {res["source"].nunique()} TFs | {res["cell_type"].nunique()} cell types')
ex_ct = ct_order(lineages=['TEC'], present=res['cell_type'])[0]
print(f'\nexample: the steroid receptors in {ex_ct}, adult vs infant')
display(res[res['source'].isin(STEROID_TFS) & (res['cell_type'] == ex_ct) & (res['contrast'] == 'adult_vs_infant')]
        .set_index('source').reindex(STEROID_TFS).round(3))


loaded cached scores from /nfs/team205/vk8/projects/thymus_ageing_atlas/General_analysis/notebooks/hormone2cell/_builders/../final_versions/github/results/thyAgeing_CollecTRI_ageDE_taa_l2_v2_scores_perCellType.csv
32,509 rows | 766 TFs | 26 cell types

example: the steroid receptors in cTEC, adult vs infant


,cell_type,contrast,n_genes,n_targets,score,r,null_mean,null_sd,z,p_z,p_emp,padj,pct_z,pct_score,n_pool
source,,,,,,,,,,,,,,,
AR,cTEC,adult_vs_infant,20012,429,4.382,0.031,0.150,1.031,4.106,0.000,0.001,0.000,97.009,97.863,702.0
NR3C1,cTEC,adult_vs_infant,20012,227,6.537,0.046,0.195,0.979,6.479,0.000,0.001,0.000,100.000,100.000,702.0
PGR,cTEC,adult_vs_infant,20012,105,4.195,0.030,0.111,1.087,3.757,0.000,0.002,0.001,96.011,97.293,702.0
ESR1,cTEC,adult_vs_infant,20012,514,3.091,0.022,0.226,1.023,2.801,0.005,0.003,0.013,91.311,93.162,702.0
ESR2,cTEC,adult_vs_infant,20012,167,2.407,0.017,0.028,0.991,2.399,0.016,0.021,0.033,88.746,88.177,702.0


In [10]:
# Null QC: `null_sd` should sit near 1 for every regulon (a t-value of a permuted vector is ~N(0,1)
# whatever the gene count), so what the permutation supplies is the CENTRE: `null_mean` is where an
# abundance-matched random gene set scores, and it varies by regulon and cell type.
print('null_sd  :', res['null_sd'].describe(percentiles=[.01, .5, .99]).round(3)[['1%', '50%', '99%']].to_dict())
print('null_mean:', res['null_mean'].describe(percentiles=[.01, .5, .99]).round(3)[['1%', '50%', '99%']].to_dict())
n_src = res.groupby(['cell_type', 'contrast'], observed=True).size()
print(f'regulons scoreable per cell type x contrast: {n_src.min()} - {n_src.max()} (median {int(n_src.median())})')
summary = res.groupby('contrast', observed=True).apply(
    lambda d: pd.Series({'cell types': d['cell_type'].nunique(), 'rows': len(d),
                         'padj < 0.05': int((d['padj'] < 0.05).sum()),
                         'fraction z > 0': round(float((d['z'] > 0).mean()), 3)}), include_groups=False)
display(summary)


null_sd  : {'1%': 0.776, '50%': 1.003, '99%': 1.185}
null_mean: {'1%': -0.919, '50%': 0.013, '99%': 0.782}
regulons scoreable per cell type x contrast: 514 - 766 (median 676)


,cell types,rows,padj < 0.05,fraction z > 0
contrast,,,,
adult_vs_infant,26.0,17116.0,2344.0,0.629
paed_vs_infant,23.0,15393.0,1239.0,0.624


In [11]:
# Reproducibility check against the scoring run used for the manuscript (internal path; skipped
# when the file is not available). The two runs share code, seed and permutation order.
REF_CSV = ('/nfs/team205/vk8/projects/thymus_ageing_atlas/General_analysis/results/scRNAseq/steroids_dea_v3/'
           f'thyAgeing_scRNAseq_steroidDEA_perCellType_{LEVEL}.csv')
if os.path.exists(REF_CSV) and N_PERM == 1000:
    ref = pd.read_csv(REF_CSV)
    ref = ref[ref['net'] == 'CollecTRI']
    m = res.merge(ref, on=['cell_type', 'contrast', 'source'], suffixes=('', '_ref'))
    print(f'{len(m):,} of {len(res):,} rows matched to the reference table')
    for col in ['score', 'r', 'null_mean', 'null_sd', 'z', 'pct_z']:
        print(f'  {col:10s} max |difference| = {np.nanmax(np.abs(m[col] - m[col + "_ref"])):.2e}')
    np.testing.assert_allclose(m['z'], m['z_ref'], atol=1e-3, rtol=1e-4)
    print('identical to the reference run')
else:
    print('reference table not available (or N_PERM != 1000) - check skipped')


32,509 of 32,509 rows matched to the reference table
  score      max |difference| = 7.19e-14
  r          max |difference| = 5.00e-16
  null_mean  max |difference| = 0.00e+00
  null_sd    max |difference| = 0.00e+00
  z          max |difference| = 7.46e-14
  pct_z      max |difference| = 0.00e+00
identical to the reference run


## 5. The steroid receptors among all TFs

Two views of the same numbers. **(a)** For every cell type, the distribution of `z` over all
CollecTRI TFs scoreable in it (violin + rug), with the five receptors drawn on top — filled markers
are FDR < 0.05. A receptor at the upper edge of its violin is among the TFs whose activity increases
most with age in that population. **(b)** The receptor percentiles themselves: one point per cell
type (coloured by lineage), one panel per contrast, so "NR3C1 sits at the 95th–100th percentile in
every stromal population" can be read directly. Both are drawn for the stroma and for all cell types.


In [12]:
all_ct    = ct_order(present=res['cell_type'])
stroma_ct = ct_order(lineages=STROMA_L1, present=res['cell_type'])
print(f'{len(all_ct)} cell types; {len(stroma_ct)} stromal: {", ".join(stroma_ct)}')

rec = res[res['source'].isin(STEROID_TFS)].copy()
rec.to_csv(f'{RESULTS}/{PREFIX}_steroidReceptors_table.csv', index=False)
for contrast in CONTRASTS:
    sub = rec[rec['contrast'] == contrast]
    here = [c for c in all_ct if c in set(sub['cell_type'])]
    piv = sub.pivot(index='source', columns='cell_type', values='pct_z').reindex(STEROID_TFS)[here]
    sig = sub.pivot(index='source', columns='cell_type', values='padj').reindex(STEROID_TFS)[here] < 0.05
    print(f'\n{contrast} - percentile among all CollecTRI TFs (pct_z); * = FDR < 0.05')
    display(piv.map(lambda v: f'{v:.0f}' if np.isfinite(v) else '') + sig.map(lambda s: '*' if s else ''))

med = (rec[rec['cell_type'].isin(stroma_ct)].groupby(['source', 'contrast'])['pct_z'].median()
       .unstack('contrast').reindex(STEROID_TFS)[CONTRASTS])
n_sig = (rec[rec['cell_type'].isin(stroma_ct)].assign(sig=lambda d: d['padj'] < 0.05)
         .groupby(['source', 'contrast'])['sig'].sum().unstack('contrast').reindex(STEROID_TFS)[CONTRASTS])
print(f'\nmedian percentile across the {len(stroma_ct)} stromal populations, and populations with FDR < 0.05:')
display(pd.concat({'median pct_z (stroma)': med.round(1), 'n populations FDR<0.05': n_sig}, axis=1))


26 cell types; 9 stromal: cTEC, mcTEC, mTEC, TEC-EMT, Fb, EC-blood, EC-lymphatic, SMC, Pericyte



paed_vs_infant - percentile among all CollecTRI TFs (pct_z); * = FDR < 0.05


cell_type,T_DN(early),T_DN,T_DP,T_αβT(entry),T_mature,T_recirc,T_innate,NK,cTEC,mcTEC,mTEC,TEC-EMT,Fb,EC-blood,SMC,Pericyte,B_cell,B_plasma,Mac,DC1,DC2,aDC,pDC
source,,,,,,,,,,,,,,,,,,,,,,,
AR,89*,90,63,95,87,97*,86,94*,98*,98*,77,100*,76,31,56,91*,97*,98*,88*,94*,96*,92*,85*
NR3C1,60,46,39,37,50,66,50,72,66,69,40,99*,100*,89,80,48,22,84,59,15,40,54,21
PGR,95*,79,85,94,89,86,95,90*,94*,99*,98*,97*,100*,51,90,92*,62,78,63,93*,83,61,69
ESR1,62,56,9,59,43,63,41,78,76,79,38,98*,68,82,84,97*,75,76,80*,98*,96*,96*,80*
ESR2,75,86,68,75,64,53,70,74,81,93,28,93,59,82,64,30,94,86,77,93,96,93,80



adult_vs_infant - percentile among all CollecTRI TFs (pct_z); * = FDR < 0.05


cell_type,T_DN(early),T_DN,T_DP,T_αβT(entry),T_mature,T_recirc,T_innate,NK,cTEC,mcTEC,mTEC,TEC-EMT,Fb,EC-blood,EC-lymphatic,SMC,Pericyte,B_cell,B_plasma,Myeloid_dev,Mono,Mac,DC1,DC2,aDC,pDC
source,,,,,,,,,,,,,,,,,,,,,,,,,,
AR,61,85*,36,58,90,99*,84*,58,97*,99*,84,99*,97*,89,70,96*,98*,65,99*,96*,95*,96*,96*,97*,98*,97*
NR3C1,94*,96*,89*,87,94*,93*,94*,99*,100*,100*,98*,100*,99*,98*,100*,98*,99*,68,92*,64,95*,91*,67,88*,95*,94*
PGR,82*,89*,75,72,85,90*,83*,65,96*,96*,99*,98*,93*,39,94*,85*,93*,25,40,72,84*,86*,79,94*,45,98*
ESR1,58,41,36,27,48,86,72,68,91*,61,9*,97*,98*,63,80,98*,97*,6*,94*,57,88*,85*,90*,85,48,60
ESR2,79*,89*,91*,66,61,38,84*,76,89*,93*,61,94*,83*,97*,70,98*,90*,55,92*,97*,92*,71*,90*,55,88*,73



median percentile across the 9 stromal populations, and populations with FDR < 0.05:


median pct_z (stroma)                 n populations FDR<0.05                
contrast        paed_vs_infant adult_vs_infant         paed_vs_infant adult_vs_infant
source                                                                               
AR                        84.0            97.0                      4               6
NR3C1                     74.7            98.9                      2               9
PGR                       95.4            94.0                      6               8
ESR1                      80.4            91.3                      2               6
ESR2                      72.2            89.8                      0               7

In [13]:
def _robust_lims(values, markers, q=(0.002, 0.998), pad=0.05):
    v = values[np.isfinite(values)]
    lo, hi = np.nanquantile(v, q)
    m = markers[np.isfinite(markers)]
    if m.size:                                           # never clip a highlighted marker out
        lo, hi = min(lo, m.min()), max(hi, m.max())
    d = pad * (hi - lo)
    return lo - d, hi + d


def plot_tf_violins(res, cts, style, value='z', padj_col='padj', contrasts=CONTRASTS,
                    width_mm='auto', panel_h_mm=38.0, per_ct_mm=6.0, pad_mm=18.0,
                    ylabel='z vs abundance-matched null', source_label='TFs', title=None,
                    marker_pt=2.8, point_pt=0.6):
    """Where the highlighted regulons sit in the distribution of ALL regulons of each cell type.

    Cell types on x, one panel per contrast (stacked, shared axes). Per cell type: a violin and a
    rug of `value` over every regulon scoreable there; the highlighted regulons (`style` =
    {source: (colour, marker, label)}) drawn on top, filled when `padj_col` < 0.05. Cell types not
    tested in a contrast are marked "n/t". Sized in mm: `width_mm='auto'` = pad + per_ct_mm per cell
    type, capped at Nature's double-column width.
    """
    present = [c for c in cts if c in set(res['cell_type'])]
    srcs = [s for s in style if s in set(res['source'])]
    if width_mm == 'auto':
        width_mm = float(np.clip(pad_mm + per_ct_mm * len(present), 45.0, NATURE_W['double']))
    key_labels = [style[s][2] for s in srcs] + ['FDR < 0.05', 'n.s.', f'all {source_label}']
    n_keys = len(key_labels)

    def _legend_mm(ncol):                                # fig.legend does not wrap: size the columns
        rows = int(np.ceil(n_keys / ncol))              # from the labels (column-major fill)
        cols = [key_labels[i * rows:(i + 1) * rows] for i in range(ncol)]
        return sum(1.1 * max(len(l) for l in c) + 7.0 for c in cols if c)

    ncol = max([n for n in range(1, n_keys + 1) if _legend_mm(n) <= width_mm - 6.0] or [1])
    legend_rows = int(np.ceil(n_keys / ncol))
    lab_mm = 1.25 * max(len(str(c)) for c in present) + 3.0
    h_mm = len(contrasts) * panel_h_mm + lab_mm + 10.0 + 5.0 * legend_rows
    fig = mm_fig(width_mm, h_mm)
    axes = np.atleast_1d(fig.subplots(len(contrasts), 1, sharex=True, sharey=True))
    pos = {c: i for i, c in enumerate(present)}

    for ax, contrast in zip(axes, contrasts):
        sub = res[res['contrast'] == contrast]
        data, at = [], []
        for c in present:
            v = sub.loc[sub['cell_type'] == c, value].to_numpy()
            v = v[np.isfinite(v)]
            if v.size:
                data.append(v)
                at.append(pos[c])
                ax.scatter(np.full(v.size, pos[c]), v, s=point_pt ** 2, c='0.35', alpha=0.6,
                           linewidths=0, zorder=2, rasterized=True)
            else:
                ax.text(pos[c], 0, 'n/t', ha='center', va='center', fontsize=6, color='0.5')
        parts = ax.violinplot(data, positions=at, showextrema=False, widths=0.86)
        for pc in parts['bodies']:
            pc.set_facecolor('#dedede'); pc.set_edgecolor('#9e9e9e'); pc.set_linewidth(0.4)
            pc.set_alpha(0.6); pc.set_zorder(1)
        ax.axhline(0, color='0.55', lw=0.5, zorder=0)
        for c in present:
            d = sub[sub['cell_type'] == c].set_index('source')
            for s in srcs:
                if s not in d.index or not np.isfinite(d.loc[s, value]):
                    continue
                col, mk, _ = style[s]
                sig = d.loc[s, padj_col] < 0.05
                ax.scatter(pos[c], d.loc[s, value], marker=mk,
                           s=(marker_pt * (1.4 if mk == '*' else 1.0)) ** 2, zorder=3,
                           facecolor=(*to_rgb(col), 0.75) if sig else 'none', edgecolor=col,
                           linewidths=0.5)
        n_src = int(sub[sub['cell_type'].isin(present)].groupby('cell_type', observed=True).size().max())
        ax.set_title(f'{contrast.replace("_", " ")}  (\u2264 {n_src:,} {source_label})')
        ax.set_ylabel(ylabel)
        ax.grid(axis='y', color='0.92', lw=0.4)
        ax.set_axisbelow(True)

    sub = res[res['cell_type'].isin(present) & res['contrast'].isin(contrasts)]
    axes[0].set_ylim(*_robust_lims(sub[value].to_numpy(), sub.loc[sub['source'].isin(srcs), value].to_numpy()))
    axes[-1].set_xticks(range(len(present)))
    axes[-1].set_xticklabels(present, rotation=90)
    axes[-1].set_xlim(-0.7, len(present) - 0.3)

    key = 4.5
    handles = [Line2D([], [], color=style[s][0], marker=style[s][1], linestyle='none',
                      markersize=key * (1.4 if style[s][1] == '*' else 1.0), label=style[s][2])
               for s in srcs]
    handles += [Line2D([], [], color='0.3', marker='o', linestyle='none', markersize=key, label='FDR < 0.05'),
                Line2D([], [], color='0.3', marker='o', linestyle='none', markersize=key,
                       markerfacecolor='none', label='n.s.'),
                Line2D([], [], color='0.35', marker='o', linestyle='none', markersize=2, alpha=0.6,
                       label=f'all {source_label}')]
    fig.legend(handles=handles, frameon=False, ncol=ncol, loc='outside lower center',
               handletextpad=0.3, columnspacing=1.0)
    if title:
        fig.suptitle(title, wrap=True)
    return fig


def plot_pct_strip(res, sources, cts, pct_col='pct_z', labels=None, contrasts=CONTRASTS,
                   width_mm=NATURE_W['single'], panel_h_mm=42.0, title=None, ylabel=None,
                   colors=None):
    """Percentile of each highlighted regulon across cell types: regulons on x, percentile on y,
    one point per cell type (coloured by lineage), the median as a bar, one panel per contrast."""
    labels = labels or {}
    lin_present = list(dict.fromkeys(LINEAGE.get(c, '?') for c in cts))
    lab_mm = 1.25 * max(len(labels.get(s, s)) for s in sources) + 2.0
    n_keys = len(lin_present)
    ncol = max(1, min(n_keys, int(width_mm // 20)))
    h_mm = panel_h_mm + lab_mm + 8.0 + 5.0 * int(np.ceil(n_keys / ncol))
    fig = mm_fig(width_mm, h_mm)
    axes = np.atleast_1d(fig.subplots(1, len(contrasts), sharey=True))
    order = {c: i for i, c in enumerate(cts)}
    for ax, contrast in zip(axes, contrasts):
        sub = res[(res['contrast'] == contrast) & res['cell_type'].isin(cts) & res['source'].isin(sources)]
        for i, s in enumerate(sources):
            d = sub[sub['source'] == s].copy()
            d = d[np.isfinite(d[pct_col])]
            if d.empty:
                ax.text(i, 50, 'n/a', ha='center', va='center', fontsize=6, color='0.5')
                continue
            vals = d[pct_col].to_numpy()
            if len(vals) >= 3:
                parts = ax.violinplot([vals], positions=[i], showextrema=False, widths=0.8)
                for pc in parts['bodies']:
                    pc.set_facecolor('#e6e6e6'); pc.set_edgecolor('none'); pc.set_alpha(0.9); pc.set_zorder(1)
            d['x'] = i + np.linspace(-0.26, 0.26, len(d))[np.argsort(np.argsort(d['cell_type'].map(order)))]
            cols = colors if colors is not None else [LINEAGE_COLORS.get(LINEAGE.get(c, '?'), '0.4') for c in d['cell_type']]
            ax.scatter(d['x'], vals, c=cols, s=10, edgecolor='white', linewidths=0.3, zorder=3)
            ax.hlines(np.median(vals), i - 0.32, i + 0.32, color='k', lw=0.9, zorder=4)
        ax.axhline(50, color='0.6', lw=0.5, ls=':', zorder=0)
        ax.set_ylim(-3, 103)
        ax.set_xticks(range(len(sources)))
        ax.set_xticklabels([labels.get(s, s) for s in sources], rotation=90)
        ax.set_xlim(-0.6, len(sources) - 0.4)
        ax.set_title(contrast.replace('_', ' '))
        ax.grid(axis='y', color='0.92', lw=0.4)
        ax.set_axisbelow(True)
    axes[0].set_ylabel(ylabel or f'percentile among all regulons ({pct_col})')
    handles = [Line2D([], [], color=LINEAGE_COLORS.get(l, '0.4'), marker='o', linestyle='none',
                      markersize=4, label=l) for l in lin_present]
    handles += [Line2D([], [], color='k', lw=0.9, label='median')]
    fig.legend(handles=handles, frameon=False, ncol=ncol + 1, loc='outside lower center',
               handletextpad=0.3, columnspacing=1.0)
    if title:
        fig.suptitle(title)
    return fig


In [14]:
style = {tf: (*RECEPTOR_STYLE[tf], tf) for tf in STEROID_TFS}
for scope, cts in [('stroma', stroma_ct), ('allCellTypes', all_ct)]:
    fig = plot_tf_violins(res, cts, style, source_label='CollecTRI TFs',
                          title=f'Steroid receptors among CollecTRI TFs - {scope}')
    save_fig(fig, f'{PREFIX}_steroidReceptors_violins_{scope}')
    plt.show()


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_steroidReceptors_violins_stroma.pdf  (72 x 119 mm)


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_steroidReceptors_violins_allCellTypes.pdf  (174 x 109 mm)


In [15]:
for scope, cts, w in [('stroma', stroma_ct, NATURE_W['single']), ('allCellTypes', all_ct, NATURE_W['two-thirds'])]:
    fig = plot_pct_strip(res, STEROID_TFS, cts, pct_col='pct_z', width_mm=w,
                         ylabel='percentile among all CollecTRI TFs (z)',
                         title=f'Steroid receptor percentiles across {scope} ({len(cts)} populations)')
    save_fig(fig, f'{PREFIX}_steroidReceptors_percentiles_{scope}')
    plt.show()


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_steroidReceptors_percentiles_stroma.pdf  (89 x 63 mm)


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_steroidReceptors_percentiles_allCellTypes.pdf  (120 x 68 mm)


## 6. Regulons that move consistently across the stroma: top 20 up and top 20 down

"Shared" is defined by **consistency of rank, not by magnitude**: every regulon is ranked by the
**median of its within-cell-type percentile** over the stromal population × contrast slots. A
percentile is a rank inside one cell type, so it is comparable across populations by construction;
ranking on mean `z` would instead favour the populations with the largest gene universes.

Two filters: a regulon must be scoreable in at least 80% of the slots (a regulon scoreable in only
two populations can trivially reach a median of 99), and must have at least 15 measured targets
(very small regulons are curation artefacts). The figure shows the 20 highest and the 20 lowest
median percentiles as two row blocks, coloured by `z` (diverging, centred on 0, symmetric limits at
the 99th percentile of |z|), with an asterisk where permutation FDR < 0.05, grey where the
population was not tested in that contrast.


In [16]:
def shared_ranking(res, cts, contrasts=CONTRASTS, pct_col='pct_z', min_frac=0.8, min_targets=15):
    """Median within-cell-type percentile over the cell type x contrast slots of `cts`."""
    d = res[res['cell_type'].isin(cts) & res['contrast'].isin(contrasts)]
    n_slots = d.groupby('source').size()
    full = int(n_slots.max())
    d = d[d['source'].isin(n_slots.index[n_slots >= min_frac * full])]
    g = d.groupby('source')
    r = pd.DataFrame({'median_pct': g[pct_col].median(), 'mean_z': g['z'].mean(), 'mean_r': g['r'].mean(),
                      'n_slots': g.size(), 'n_sig': g['padj'].apply(lambda p: int((p < 0.05).sum())),
                      'n_targets': g['n_targets'].max()})
    r['kept'] = r['n_targets'] >= min_targets
    return r.sort_values('median_pct', ascending=False), full


def _matrix(res, sources, cts, contrast, value):
    m = (res[res['contrast'] == contrast]
         .pivot_table(index='source', columns='cell_type', values=value, observed=True))
    return m.reindex(index=sources, columns=cts)


def plot_shared_heatmap(res, cts, up, dn, contrasts=CONTRASTS, value='z', width_mm=NATURE_W['double'],
                        row_mm=3.1, base_pt=7.0, qlim=0.99, star_at=0.05, labels=None, title=None,
                        block_names=('up with age', 'down with age')):
    """Two row blocks (up / down) x cell types, one panel per contrast; colour = `value`."""
    labels = labels or {}
    blocks = [(block_names[0], list(up)), (block_names[1], list(dn))]
    mats = {(b, c): _matrix(res, s, cts, c, value) for b, s in blocks for c in contrasts}
    padj = {(b, c): _matrix(res, s, cts, c, 'padj') for b, s in blocks for c in contrasts}
    v = np.concatenate([m.to_numpy().ravel() for m in mats.values()])
    v = v[np.isfinite(v)]
    vmax = max(float(np.nanquantile(np.abs(v), qlim)), 1e-9)
    n_rows = len(up) + len(dn)
    lab_mm = 1.05 * max(len(str(c)) for c in cts) + 4.0
    h_mm = (n_rows + 1.4) * row_mm + lab_mm + 12.0
    with plt.rc_context({'font.size': base_pt, 'xtick.labelsize': base_pt, 'ytick.labelsize': base_pt,
                         'axes.titlesize': base_pt + 1}):
        fig = mm_fig(width_mm, h_mm)
        gs = fig.add_gridspec(2, len(contrasts) + 1, height_ratios=[len(up), len(dn)],
                              width_ratios=[1] * len(contrasts) + [0.035], hspace=1.4 / n_rows, wspace=0.03)
        for bi, (bname, srcs) in enumerate(blocks):
            for ci, c in enumerate(contrasts):
                ax = fig.add_subplot(gs[bi, ci])
                sns.heatmap(mats[(bname, c)], ax=ax, cmap=DIVERGING, center=0, vmin=-vmax, vmax=vmax,
                            cbar=False, linewidths=0,
                            yticklabels=[labels.get(s, s) for s in srcs] if ci == 0 else False,
                            xticklabels=(bi == 1))
                ax.grid(False)
                ax.set_xlabel('')
                ax.set_ylabel(bname if ci == 0 else '', labelpad=2)
                ax.tick_params(length=1.2, width=0.4, pad=1.2)
                if bi == 0:
                    ax.set_title(c.replace('_', ' '), pad=3)
                if bi == 1:
                    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
                P = padj[(bname, c)]
                for yi in range(P.shape[0]):
                    for xi in range(P.shape[1]):
                        p = P.iat[yi, xi]
                        if np.isfinite(p) and p < star_at:
                            ax.text(xi + 0.5, yi + 0.62, '*', ha='center', va='center', fontsize=base_pt - 1.5)
        cax = fig.add_subplot(gs[:, -1])
        fig.colorbar(ScalarMappable(Normalize(-vmax, vmax), DIVERGING), cax=cax,
                     label='z vs abundance-matched null')
        cax.tick_params(length=1.5, width=0.4, pad=1.5)
        if title:
            fig.suptitle(title)
    return fig


In [17]:
ranking, n_slots_full = shared_ranking(res, stroma_ct, pct_col='pct_z', min_frac=0.8, min_targets=15)
ranking.to_csv(f'{RESULTS}/{PREFIX}_sharedRanking_stroma.csv')
ok = ranking[ranking['kept']]
up20, dn20 = list(ok.head(20).index), list(ok.tail(20).index[::-1])
print(f'{len(stroma_ct)} stromal populations = {n_slots_full} population x contrast slots; '
      f'{len(ranking)} TFs scoreable in >= 80% of them, {len(ok)} with >= 15 measured targets')

fig = plot_shared_heatmap(res, stroma_ct, up20, dn20,
                          title='CollecTRI regulons most consistently up / down with age across the stroma')
save_fig(fig, f'{PREFIX}_top20_sharedStroma')
plt.show()

shared20 = ranking.loc[up20 + dn20].assign(direction=['up'] * len(up20) + ['down'] * len(dn20))
shared20.to_csv(f'{RESULTS}/{PREFIX}_top20_sharedStroma.csv')
cols = ['direction', 'median_pct', 'mean_z', 'mean_r', 'n_sig', 'n_slots', 'n_targets']
display(shared20[cols].round(2))
print('steroid receptors in the UP list:  ', [s for s in STEROID_TFS if s in up20])
print('steroid receptors in the DOWN list:', [s for s in STEROID_TFS if s in dn20])


9 stromal populations = 17 population x contrast slots; 677 TFs scoreable in >= 80% of them, 467 with >= 15 measured targets


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_top20_sharedStroma.pdf  (183 x 157 mm)


,direction,median_pct,mean_z,mean_r,n_sig,n_slots,n_targets
source,,,,,,,
BMAL1,up,99.58,4.64,0.03,16,17,57
STAT3,up,99.13,3.97,0.03,14,17,429
NR3C1,up,98.33,3.41,0.02,11,17,288
FOXO3,up,98.30,3.58,0.03,15,17,195
CLOCK,up,97.63,3.77,0.03,16,17,66
NCOA1,up,96.18,3.17,0.02,14,17,91
SMAD2,up,95.77,3.11,0.02,11,17,106
MEF2D,up,95.66,2.83,0.02,13,17,29
AR,up,95.66,2.80,0.02,10,17,505


steroid receptors in the UP list:   ['AR', 'NR3C1', 'PGR']
steroid receptors in the DOWN list: []


## 7. Control: are the steroid-receptor signals just the generic stress programme?

Steroid receptors share many targets with the immediate-early / stress-response machinery
(FOS/JUN, KLF, DUSP1, SGK1, PER1, …), and dissociated tissue carries a stress signature. So a
receptor could rank high simply because its regulon contains stress genes that rise with age. Three
tests, each stricter than the last, on the stromal populations:

1. **Do the stress TFs themselves rank high?** If the age signal were generic stress, the regulons
   of AP-1 (JUN/FOS/ATF3), NF-κB, NRF2/HIF1A/TP53 or the UPR/heat-shock factors would sit at the top
   of the TF distribution, above the receptors.
2. **How much of each receptor's score is carried by stress-signature targets?** The ULM correlation
   decomposes exactly into per-gene contributions, `c_i = (w_i − w̄)(y_i − ȳ) / ((n−1)·sd(w)·sd(y))`,
   which sum to `r`. The share of `r` carried by stress genes, divided by their share of the
   regulon's targets, is an enrichment: ≈ 1 means the stress genes are passengers, ≫ 1 that they
   drive the signal.
3. **Does the receptor's rank survive removing the stress genes?** (a) The stress genes are deleted
   from the gene universe and everything is re-scored, permutation null included, so both the regulon
   and its background lose them. (b) The stress-free half of each receptor regulon is scored as a
   separate regulon on the full universe and its percentile is read against all TFs.

Stress definitions: **AP-1 regulon targets** = the CollecTRI targets of JUN/JUNB/JUND/FOS/FOSB/
FOSL1/FOSL2/ATF3 (same resource and curation as the receptor regulons, so directly comparable), and
**broad stress** = those ∪ curated immediate-early, heat-shock and dissociation-induced gene lists ∪
eight MSigDB Hallmark stress programmes (TNFα/NF-κB, inflammatory response, hypoxia, UPR, p53,
apoptosis, ROS, IL6/JAK/STAT3). The broad set is deliberately aggressive.


In [18]:
AP1_TFS = ['JUN', 'JUNB', 'JUND', 'FOS', 'FOSB', 'FOSL1', 'FOSL2', 'ATF3']
STRESS_TF_GROUPS = {'AP-1': ['JUN', 'JUNB', 'FOS', 'FOSL2', 'ATF3'],
                    'NF-kB': ['NFKB1', 'RELA', 'RELB'],
                    'oxidative / damage': ['NFE2L2', 'HIF1A', 'TP53', 'EGR1'],
                    'UPR / heat shock': ['ATF4', 'DDIT3', 'XBP1', 'HSF1']}
# immediate-early / early stress-responsive genes (note PER1, SGK1, KLF9, CEBPD are also canonical
# glucocorticoid targets - that ambiguity is what this control quantifies)
IEG = ['FOS', 'FOSB', 'FOSL1', 'FOSL2', 'JUN', 'JUNB', 'JUND', 'ATF3', 'JDP2',
       'EGR1', 'EGR2', 'EGR3', 'EGR4', 'IER2', 'IER3', 'IER5', 'MIDN',
       'NR4A1', 'NR4A2', 'NR4A3', 'DUSP1', 'DUSP2', 'DUSP5', 'DUSP6',
       'ZFP36', 'ZFP36L1', 'ZFP36L2', 'KLF2', 'KLF4', 'KLF6', 'KLF9', 'KLF10',
       'SOCS3', 'BTG1', 'BTG2', 'PPP1R15A', 'GADD45B', 'GADD45G', 'CCN1', 'CCN2',
       'RHOB', 'MCL1', 'NFKBIA', 'NFKBIZ', 'TNFAIP3', 'PER1', 'SGK1', 'SIK1',
       'MYC', 'ID1', 'ID2', 'ID3', 'CEBPB', 'CEBPD', 'TRIB1', 'PLK2', 'SERTAD1', 'PNRC1']
HEAT_SHOCK = ['HSPA1A', 'HSPA1B', 'HSPA4', 'HSPA4L', 'HSPA5', 'HSPA6', 'HSPA8', 'HSPA9',
              'HSPB1', 'HSPB8', 'HSPD1', 'HSPE1', 'HSPH1', 'HSP90AA1', 'HSP90AB1', 'HSP90B1',
              'DNAJA1', 'DNAJA4', 'DNAJB1', 'DNAJB4', 'DNAJB6', 'DNAJB9', 'BAG3', 'AHSA1',
              'CHORDC1', 'CRYAB', 'SERPINH1', 'ST13', 'CACYBP', 'UBB', 'UBC', 'HSPBP1',
              'FKBP4', 'PTGES3', 'STIP1']
# van den Brink et al. 2017 (Nat Methods) dissociation-induced signature, human symbols
DISSOCIATION = ['FOS', 'FOSB', 'JUN', 'JUNB', 'JUND', 'EGR1', 'ATF3', 'IER2', 'IER3', 'DUSP1',
                'ZFP36', 'SOCS3', 'KLF2', 'KLF4', 'KLF6', 'NFKBIA', 'PPP1R15A', 'RHOB',
                'CEBPB', 'CEBPD', 'ID1', 'ID2', 'ID3', 'BTG1', 'BTG2', 'GADD45B', 'NR4A1',
                'SGK1', 'CCN1', 'MT1X', 'MT2A', 'HSPA1A', 'HSPA1B', 'HSPA8', 'HSPB1', 'HSPH1',
                'HSP90AA1', 'HSP90AB1', 'DNAJA1', 'DNAJB1', 'HSPE1', 'UBC', 'MYC', 'IRF1',
                'CXCL1', 'CXCL2', 'ICAM1', 'PLK2', 'EIF1', 'SRSF5', 'SRSF7', 'PNRC1']
# annotation only (never used to select): canonical direct glucocorticoid / androgen targets
GR_CANONICAL = ['FKBP5', 'TSC22D3', 'PER1', 'KLF9', 'DDIT4', 'SGK1', 'ZBTB16', 'IRS2', 'SESN1',
                'THBD', 'CEBPD', 'MT1X', 'MT2A', 'SCNN1A', 'ANGPTL4', 'TXNIP', 'PLIN2', 'GLUL',
                'CIDEC', 'AZGP1', 'PDK4', 'ALOX15B', 'AMPD3', 'ADAMTS1', 'TIPARP', 'ETS2',
                'DUSP1', 'NFKBIA', 'BIRC3', 'FOXO3', 'CXCR4', 'SLC19A2']


def read_gmt(path):
    out = {}
    with open(path) as fh:
        for line in fh:
            parts = line.rstrip('\n').split('\t')
            if len(parts) > 2:
                out[parts[0]] = set(parts[2:])
    return out


hallmark = read_gmt(f'{DATA_DIR}/msigdb_hallmark_stress_2025.1.Hs.gmt')
print('Hallmark stress programmes:', ', '.join(f'{k} ({len(v)})' for k, v in hallmark.items()))
ap1_targets = set(net.loc[net['source'].isin(AP1_TFS), 'target'])
STRESS_SETS = {'AP-1 regulon targets': ap1_targets & measured,
               'broad stress': (set(IEG) | set(HEAT_SHOCK) | set(DISSOCIATION) | ap1_targets
                                | set().union(*hallmark.values())) & measured}
for k, v in STRESS_SETS.items():
    print(f'{k}: {len(v):,} measured genes')

rows = []
for tf in STEROID_TFS + ['JUN', 'FOS', 'NFKB1']:
    t = set(net.loc[net['source'] == tf, 'target']) & measured
    rows.append({'TF': tf, 'measured targets': len(t),
                 **{f'% in {k}': round(100 * len(t & v) / len(t), 1) for k, v in STRESS_SETS.items()}})
print('\nshare of each regulon that lies inside the stress definitions:')
display(pd.DataFrame(rows).set_index('TF'))


Hallmark stress programmes: HALLMARK_TNFA_SIGNALING_VIA_NFKB (200), HALLMARK_INFLAMMATORY_RESPONSE (200), HALLMARK_HYPOXIA (200), HALLMARK_UNFOLDED_PROTEIN_RESPONSE (113), HALLMARK_P53_PATHWAY (200), HALLMARK_APOPTOSIS (161), HALLMARK_REACTIVE_OXYGEN_SPECIES_PATHWAY (49), HALLMARK_IL6_JAK_STAT3_SIGNALING (87)
AP-1 regulon targets: 853 measured genes
broad stress: 1,614 measured genes

share of each regulon that lies inside the stress definitions:


,measured targets,% in AP-1 regulon targets,% in broad stress
TF,,,
AR,506,34.0,43.5
NR3C1,288,51.4,62.5
PGR,130,47.7,53.8
ESR1,622,37.5,43.7
ESR2,206,39.8,47.6
JUN,652,100.0,100.0
FOS,340,100.0,100.0
NFKB1,426,48.8,61.5


In [19]:
# 1. the stress TFs' own regulons, ranked the same way as the receptors (median percentile over the
#    stromal populations)
ref_tfs = [t for g in STRESS_TF_GROUPS.values() for t in g if t in net_sources]
group_of = {t: g for g, ts in STRESS_TF_GROUPS.items() for t in ts}
comp = STEROID_TFS + ref_tfs
tab = (res[res['cell_type'].isin(stroma_ct) & res['source'].isin(comp)]
       .groupby(['source', 'contrast'])['pct_z'].median().unstack('contrast').reindex(comp)[CONTRASTS])
tab.insert(0, 'group', [group_of.get(t, 'steroid receptor') for t in comp])
tab.to_csv(f'{RESULTS}/{PREFIX}_stressControl_TFranks_stroma.csv')
print('median percentile (pct_z) across the stromal populations:')
display(tab.round(1))

fig = plot_pct_strip(res, comp, stroma_ct, pct_col='pct_z', width_mm=NATURE_W['two-thirds'],
                     ylabel='percentile among all CollecTRI TFs (z)',
                     title='Steroid receptors vs stress-response TFs - stromal populations')
save_fig(fig, f'{PREFIX}_stressControl_TFranks_stroma')
plt.show()


median percentile (pct_z) across the stromal populations:


contrast,group,paed_vs_infant,adult_vs_infant
source,,,
AR,steroid receptor,84.0,97.0
NR3C1,steroid receptor,74.7,98.9
PGR,steroid receptor,95.4,94.0
ESR1,steroid receptor,80.4,91.3
ESR2,steroid receptor,72.2,89.8
JUN,AP-1,93.5,96.2
JUNB,AP-1,59.1,78.8
FOS,AP-1,70.1,94.1
FOSL2,AP-1,60.6,64.0


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_stressControl_TFranks_stroma.pdf  (120 x 64 mm)


In [20]:
# 2. per-gene decomposition of the ULM correlation
def weight_vector(universe, net, source):
    """The regulon's weight over `universe` (0 for non-targets) - the x decoupler regresses on."""
    w = pd.Series(0.0, index=pd.Index(universe))
    d = net[(net['source'] == source) & net['target'].isin(w.index)]
    w.loc[d['target'].to_numpy()] = d['weight'].to_numpy()
    return w.to_numpy()


def contributions(y, w):
    """Per-gene share of the Pearson correlation r(y, w); the terms sum exactly to r."""
    y, w = np.asarray(y, dtype=np.float64), np.asarray(w, dtype=np.float64)
    n = y.size
    return ((w - w.mean()) * (y - y.mean())) / ((n - 1) * w.std(ddof=1) * y.std(ddof=1))


def leading_edge(de, net, sources, cts, contrasts, min_targets=TMIN):
    """Contribution of every TARGET gene to each regulon's r, per cell type x contrast."""
    out = []
    for src in sources:
        for c in contrasts:
            for ct in cts:
                d = de[(de['cell_type'] == ct) & (de['contrast'] == c)].sort_values('gene_name')
                if d.empty:
                    continue
                uni = d['gene_name'].to_numpy()
                w = weight_vector(uni, net, src)
                if (w != 0).sum() < min_targets:
                    continue
                ci = contributions(d['stat'].to_numpy(), w)
                keep = w != 0
                out.append(pd.DataFrame({'source': src, 'contrast': c, 'cell_type': ct, 'gene': uni[keep],
                                         'contrib': ci[keep], 'stat': d['stat'].to_numpy()[keep],
                                         'weight': w[keep]}))
    return pd.concat(out, ignore_index=True)


le = leading_edge(de, net, STEROID_TFS + ['JUN', 'FOS'], stroma_ct, CONTRASTS)
le.to_csv(f'{RESULTS}/{PREFIX}_steroidReceptors_targetContributions_stroma.csv', index=False)
print(f'{len(le):,} target x population x contrast contributions')


def class_contribution(le, source, contrast, genes, r_floor=0.005):
    """Share of the target-side correlation carried by `genes`, relative to their share of targets."""
    d = le[(le['source'] == source) & (le['contrast'] == contrast)]
    per_gene = d.groupby('gene')['contrib'].mean()           # averaged over the stromal populations
    total = per_gene.sum()
    hit = per_gene.index.intersection(list(genes))
    share_t = len(hit) / len(per_gene)
    share_r = per_gene[hit].sum() / total if abs(total) >= r_floor else np.nan
    return {'n_targets': len(per_gene), 'n_in_set': len(hit), 'r_targets': total,
            'share_targets_%': 100 * share_t, 'share_r_%': 100 * share_r,
            'enrichment': share_r / share_t if share_t else np.nan}


contrib = pd.DataFrame([{'TF': s, 'contrast': c, 'stress set': k, **class_contribution(le, s, c, v)}
                        for s in STEROID_TFS + ['JUN', 'FOS'] for c in CONTRASTS for k, v in STRESS_SETS.items()])
contrib.to_csv(f'{RESULTS}/{PREFIX}_stressControl_contribution_stroma.csv', index=False)
print('enrichment = (share of r carried by stress targets) / (share of targets that are stress genes);'
      ' ~1 = passengers, >>1 = the stress genes drive the signal. Blank where |r_targets| < 0.005.')
display(contrib.pivot(index='TF', columns=['stress set', 'contrast'], values='enrichment')
        .reindex(STEROID_TFS + ['JUN', 'FOS']).round(2))
print("\npercentage of each regulon's measured targets that are stress genes:")
display(contrib.pivot(index='TF', columns=['stress set', 'contrast'], values='share_targets_%')
        .reindex(STEROID_TFS + ['JUN', 'FOS']).round(1))


38,148 target x population x contrast contributions
enrichment = (share of r carried by stress targets) / (share of targets that are stress genes); ~1 = passengers, >>1 = the stress genes drive the signal. Blank where |r_targets| < 0.005.


stress set,AP-1 regulon targets,broad stress,AP-1 regulon targets,broad stress
contrast,paed_vs_infant,paed_vs_infant,adult_vs_infant,adult_vs_infant
TF,,,,
AR,0.78,0.90,0.85,1.54
NR3C1,0.39,0.95,0.56,1.01
PGR,1.17,1.17,0.89,1.20
ESR1,NaN,NaN,2.05,2.90
ESR2,NaN,NaN,1.69,2.20
JUN,NaN,NaN,1.00,1.00
FOS,NaN,NaN,1.00,1.00



percentage of each regulon's measured targets that are stress genes:


stress set,AP-1 regulon targets,broad stress,AP-1 regulon targets,broad stress
contrast,paed_vs_infant,paed_vs_infant,adult_vs_infant,adult_vs_infant
TF,,,,
AR,34.0,43.5,34.0,43.5
NR3C1,51.4,62.5,51.4,62.5
PGR,47.7,53.8,47.7,53.8
ESR1,37.5,43.7,37.5,43.7
ESR2,39.8,47.6,39.8,47.6
JUN,100.0,100.0,100.0,100.0
FOS,100.0,100.0,100.0,100.0


In [21]:
# 3a. delete the broad stress genes from the universe and re-score the stroma (null included)
STRESS_DROP = STRESS_SETS['broad stress']
print(f'(a) re-scoring {len(stroma_ct)} stromal populations without {len(STRESS_DROP):,} stress genes')
res_drop = add_percentiles(score_all(de, net, cell_types=stroma_ct, drop_genes=STRESS_DROP), pool=net_sources)
res_drop.to_csv(f'{RESULTS}/{PREFIX}_stressControl_dropRescore_stroma.csv', index=False)

# 3b. the stress-free half of each receptor regulon as its own source, on the full universe
other = pd.concat([net[(net['source'] == s) & ~net['target'].isin(STRESS_DROP)].assign(source=s + '_nonStress')
                   for s in STEROID_TFS], ignore_index=True)
print('(b) non-stress halves:', other.groupby('source').size().to_dict())
res_split = score_all(de, pd.concat([net, other], ignore_index=True), cell_types=stroma_ct)
res_split = add_percentiles(res_split, pool=net_sources)          # the halves are NOT in the pool ...
is_half = res_split['source'].str.endswith('_nonStress')
pct_half = {}
for (ct, c), g in res_split.groupby(['cell_type', 'contrast']):    # ... their percentile is read against it
    pool_z = g.loc[g['source'].isin(net_sources), 'z'].dropna().to_numpy()
    for idx, z in g.loc[is_half.loc[g.index], 'z'].items():
        pct_half[idx] = percentileofscore(pool_z, z, kind='weak') if np.isfinite(z) else np.nan
res_split.loc[list(pct_half), 'pct_z'] = pd.Series(pct_half)
res_split.to_csv(f'{RESULTS}/{PREFIX}_stressControl_splitRegulon_stroma.csv', index=False)


(a) re-scoring 9 stromal populations without 1,614 stress genes


[1/9] cTEC: 18,621 genes, 481 regulons (4 s)


[2/9] mcTEC: 20,273 genes, 503 regulons (7 s)


[3/9] mTEC: 25,899 genes, 569 regulons (13 s)


[4/9] TEC-EMT: 15,860 genes, 501 regulons (16 s)


...

[5/9] Fb: 23,646 genes, 738 regulons (21 s)


[6/9] EC-blood: 22,126 genes, 724 regulons (26 s)


[7/9] EC-lymphatic: 13,490 genes, 637 regulons (27 s)


[8/9] SMC: 17,897 genes, 696 regulons (31 s)


[9/9] Pericyte: 15,498 genes, 674 regulons (33 s)


In [22]:
def median_pct(df, sources, rename=None):
    d = df[df['cell_type'].isin(stroma_ct) & df['source'].isin(sources)]
    out = d.groupby(['source', 'contrast'])['pct_z'].median().unstack('contrast')[CONTRASTS]
    return out.rename(index=rename) if rename else out


def n_sig(df, sources, rename=None):
    d = df[df['cell_type'].isin(stroma_ct) & df['source'].isin(sources)].assign(sig=lambda x: x['padj'] < 0.05)
    out = d.groupby(['source', 'contrast'])['sig'].sum().unstack('contrast')[CONTRASTS]
    return out.rename(index=rename) if rename else out


halves = [s + '_nonStress' for s in STEROID_TFS]
strip = lambda s: s.replace('_nonStress', '')
summary = pd.concat({'full regulon': median_pct(res, STEROID_TFS),
                     'stress genes deleted from universe': median_pct(res_drop, STEROID_TFS),
                     'non-stress half of regulon': median_pct(res_split, halves, strip)}, axis=1).reindex(STEROID_TFS)
summary.to_csv(f'{RESULTS}/{PREFIX}_stressControl_summary_stroma.csv')
print('median percentile among all CollecTRI TFs across the stromal populations:')
display(summary.round(1))
print(f'\nstromal populations (of {len(stroma_ct)}) with FDR < 0.05:')
display(pd.concat({'full regulon': n_sig(res, STEROID_TFS),
                   'non-stress half of regulon': n_sig(res_split, halves, strip)}, axis=1).reindex(STEROID_TFS))
print('\nnon-stress halves per population (pct_z; * = FDR < 0.05), adult vs infant:')
sub = res_split[(res_split['contrast'] == 'adult_vs_infant') & res_split['source'].isin(halves)]
piv = sub.pivot(index='source', columns='cell_type', values='pct_z').reindex(halves)[stroma_ct]
sig = sub.pivot(index='source', columns='cell_type', values='padj').reindex(halves)[stroma_ct] < 0.05
display(piv.map(lambda v: f'{v:.0f}' if np.isfinite(v) else '') + sig.map(lambda s: '*' if s else ''))


median percentile among all CollecTRI TFs across the stromal populations:


full regulon                 stress genes deleted from universe                 non-stress half of regulon                
contrast paed_vs_infant adult_vs_infant                     paed_vs_infant adult_vs_infant             paed_vs_infant adult_vs_infant
source                                                                                                                               
AR                 84.0            97.0                               92.5            93.6                       84.3            85.5
NR3C1              74.7            98.9                               78.8            99.0                       64.0            92.6
PGR                95.4            94.0                               94.2            93.4                       86.3            79.3
ESR1               80.4            91.3                               47.8            59.1                       35.3            33.4
ESR2               72.2            89.8                               51.9            53.4                       39.7            45.0


stromal populations (of 9) with FDR < 0.05:


full regulon                 non-stress half of regulon                
contrast paed_vs_infant adult_vs_infant             paed_vs_infant adult_vs_infant
source                                                                            
AR                    4               6                          2               4
NR3C1                 2               9                          1               7
PGR                   6               8                          1               2
ESR1                  2               6                          0               1
ESR2                  0               7                          0               0


non-stress halves per population (pct_z; * = FDR < 0.05), adult vs infant:


cell_type,cTEC,mcTEC,mTEC,TEC-EMT,Fb,EC-blood,EC-lymphatic,SMC,Pericyte
source,,,,,,,,,
AR_nonStress,76,89,68,91*,86*,72,33,92*,92*
NR3C1_nonStress,96*,96*,93,93*,93*,84,97*,87*,90*
PGR_nonStress,90*,87,97,94*,76,33,79,57,65
ESR1_nonStress,84,30,8*,68,69,4,33,66,33
ESR2_nonStress,38,36,52,45,24,69,3,73,49


### Reading the stress control (numbers from the tables above, 1,000 permutations)

* **The stress TFs' own regulons do not explain the receptors.** NF-κB (NFKB1, RELA, RELB) and
  ATF3 / FOSL2 rank in the middle or bottom of the TF distribution in the adult contrast, so the
  signal is not inflammation or dissociation stress. JUN, FOS and an oxidative / hypoxia axis
  (HIF1A, NFE2L2, TP53) do rank as high as the receptors, which is why tests 2 and 3 are needed.
* **Stress genes are passengers in the NR3C1, AR and PGR regulons but drivers in ESR1 / ESR2.**
  Adult vs infant, broad stress set: the stress targets carry 1.0× (NR3C1), 1.2× (PGR) and 1.5× (AR)
  their proportional share of the correlation, against 2.9× (ESR1) and 2.2× (ESR2) — i.e. more than
  the whole ESR1 / ESR2 signal is carried by their stress-annotated targets, the remainder pulling
  the other way. On the like-for-like AP-1 tier, AR / NR3C1 / PGR are at 0.6–0.9×, ESR1 / ESR2 at
  1.7–2.1×.
* **The receptor ranks survive removing the stress genes; ESR1 / ESR2 do not.** Median percentile
  across the stroma, adult vs infant, full regulon → stress genes deleted from the universe →
  stress-free half of the regulon: NR3C1 98.9 → 99.0 → 92.6; AR 97.0 → 93.6 → 85.5; PGR
  94.0 → 93.4 → 79.3; **ESR1 91.3 → 59.1 → 33.4; ESR2 89.8 → 53.4 → 45.0**. In the paediatric
  contrast the same pattern holds (AR 84.0 → 92.5 → 84.3; PGR 95.4 → 94.2 → 86.3; NR3C1
  74.7 → 78.8 → 64.0; ESR1 80.4 → 47.8 → 35.3; ESR2 72.2 → 51.9 → 39.7).

**Conclusion.** The AR, NR3C1 and PGR signals are not the generic stress programme: their ranks
hold when the stress genes are removed, and their leading edges (next section) are canonical
glucocorticoid / androgen targets. ESR1 and ESR2 rank high only because their regulons overlap the
stress programme, and were therefore not prioritised further.


## 8. Which target genes drive the AR, NR3C1 and PGR signals in the stroma?

The per-gene contributions `c_i` from the previous section answer this directly: a gene *drives* a
receptor's activity increase when `c_i > 0`, which happens in two ways — an **activated target
rising** with age, or a **repressed target falling**. Both arms are shown; a single "top targets"
list would be almost entirely the rising arm, because CollecTRI is predominantly activating edges.

Per receptor and contrast, each target's contribution is expressed as a percentage of the
population's target-side correlation and averaged over the stromal populations (`share_%`); genes
are ranked on that share and must be tested in at least half of the populations. The figures show
the top drivers of each arm coloured by their Wald statistic in every stromal population, with the
share as a bar (one bar scale for both arms); an open ring marks a population in which the gene does
not contribute positively. `†` marks canonical direct glucocorticoid / androgen targets, as an
annotation only. Finally the three receptors are pooled into one "steroid programme": each receptor's
per-gene shares are summed, so a gene driving several receptors ranks highest.


In [23]:
DRIVER_TFS = ['AR', 'NR3C1', 'PGR']
R_FLOOR = 0.005          # |r_targets| below this: shares would be meaningless multiples of ~0


def target_table(le, source, cts, contrast, r_floor=R_FLOOR):
    """Per gene: mean contribution, mean share of the population's target-side r, direction, support."""
    d = le[(le['contrast'] == contrast) & (le['source'] == source) & le['cell_type'].isin(cts)]
    tot = d.groupby('cell_type')['contrib'].transform('sum')
    d = d.assign(share=100 * d['contrib'] / tot.where(tot.abs() >= r_floor))
    g = d.groupby('gene')
    out = pd.DataFrame({'weight': g['weight'].first(), 'mean_contrib': g['contrib'].mean(),
                        'share_%': g['share'].mean(), 'mean_stat': g['stat'].mean(),
                        'n_pops': g['cell_type'].nunique(),
                        'n_drives': g['contrib'].apply(lambda x: int((x > 0).sum()))})
    out['arm'] = np.where(out['mean_stat'] > 0, 'rising', 'falling')
    out['canonical_GC_AR_target'] = out.index.isin(GR_CANONICAL)
    return out.sort_values('share_%', ascending=False)


def top_drivers(tt, n=15, min_pops=None):
    """The n rising and n falling targets with the largest positive share, tested in >= min_pops populations."""
    min_pops = int(np.ceil(0.5 * tt['n_pops'].max())) if min_pops is None else min_pops
    drv = tt[(tt['mean_contrib'] > 0) & (tt['share_%'] > 0) & (tt['n_pops'] >= min_pops)]
    return list(drv[drv['arm'] == 'rising'].head(n).index), list(drv[drv['arm'] == 'falling'].head(n).index)


tables = {(s, c): target_table(le, s, stroma_ct, c) for s in DRIVER_TFS for c in CONTRASTS}
tt_all = pd.concat(tables, names=['source', 'contrast', 'gene']).reset_index()
tt_all.to_csv(f'{RESULTS}/{PREFIX}_steroidReceptors_targetDrivers_stroma.csv', index=False)
for s in DRIVER_TFS:
    up, dn = top_drivers(tables[(s, 'adult_vs_infant')])
    print(f'\n{s}, adult vs infant - top drivers ({len(up)} rising / {len(dn)} falling):')
    display(tables[(s, 'adult_vs_infant')].loc[up + dn].round(3))



AR, adult vs infant - top drivers (15 rising / 15 falling):


,weight,mean_contrib,share_%,mean_stat,n_pops,n_drives,arm,canonical_GC_AR_target
gene,,,,,,,,
MT2A,1.0,0.001,5.947,5.162,9,9,rising,True
FKBP5,1.0,0.001,5.731,4.491,9,9,rising,True
ZBTB16,1.0,0.001,5.217,4.373,9,9,rising,True
ARG2,1.0,0.001,4.980,4.640,9,9,rising,False
SOD2,1.0,0.001,4.504,3.874,9,9,rising,False
HMOX1,1.0,0.001,3.567,3.050,9,9,rising,False
PTPN1,1.0,0.001,3.535,2.843,9,9,rising,False
CEBPD,1.0,0.001,3.530,2.895,9,9,rising,True
SESN1,1.0,0.000,3.400,2.385,9,9,rising,True



NR3C1, adult vs infant - top drivers (15 rising / 15 falling):


,weight,mean_contrib,share_%,mean_stat,n_pops,n_drives,arm,canonical_GC_AR_target
gene,,,,,,,,
MAOA,1.0,0.002,5.326,5.965,9,9,rising,False
MT2A,1.0,0.002,4.686,5.162,9,9,rising,True
FKBP5,1.0,0.001,4.288,4.491,9,9,rising,True
CEBPB,1.0,0.001,3.999,4.228,9,9,rising,False
PDK4,1.0,0.001,3.454,3.333,9,9,rising,True
GLUL,1.0,0.001,3.421,4.276,9,8,rising,True
MT1A,1.0,0.001,3.220,3.466,9,9,rising,False
TGFBR2,1.0,0.001,3.166,3.365,9,9,rising,False
LRRK2,1.0,0.001,3.067,2.771,9,8,rising,False



PGR, adult vs infant - top drivers (15 rising / 9 falling):


,weight,mean_contrib,share_%,mean_stat,n_pops,n_drives,arm,canonical_GC_AR_target
gene,,,,,,,,
IGFBP5,1.0,0.002,8.958,5.345,9,8,rising,False
FKBP5,1.0,0.002,8.434,4.491,9,9,rising,True
PLIN2,1.0,0.002,6.926,3.327,9,8,rising,True
PGR,1.0,0.001,6.706,2.908,7,7,rising,False
THBS1,1.0,0.001,6.584,3.121,9,8,rising,False
SLPI,1.0,0.002,6.506,3.519,9,9,rising,False
ETS2,1.0,0.002,6.383,3.562,9,9,rising,True
TIMP3,1.0,0.002,6.373,3.617,9,9,rising,False
ERRFI1,1.0,0.001,6.171,3.265,9,9,rising,False


In [24]:
def _sign_tag(w, eps=0.05):
    return '+' if w > eps else '−' if w < -eps else '±'


def plot_target_drivers(le, source, cts, genes, contrast, tt, bar_color='#7f7f7f',
                        width_mm=NATURE_W['two-thirds'], row_mm=3.4, base_pt=7.0, qlim=0.98,
                        bar_frac=0.30, bar_label='% of target-side r', title=None,
                        block_names=('targets rising with age', 'targets falling with age'),
                        arms=('rising', 'falling')):
    """Driver genes x populations coloured by the Wald statistic, with each gene's share as a bar.
    `arms` selects the blocks drawn: both, or ('rising',) for the upper half only."""
    d = le[(le['contrast'] == contrast) & (le['source'] == source) & le['cell_type'].isin(cts)]
    sign = d.drop_duplicates('gene').set_index('gene')['weight']
    blocks = [(nm, [g for g in gs if g in tt.index])
              for nm, gs, arm in zip(block_names, genes, ('rising', 'falling')) if arm in arms]
    blocks = [(nm, gs) for nm, gs in blocks if gs]
    assert blocks, f'nothing to plot for {source}'
    stat = d.pivot_table(index='gene', columns='cell_type', values='stat')
    contrib = d.pivot_table(index='gene', columns='cell_type', values='contrib')
    cts = [c for c in cts if c in stat.columns]
    allg = [g for _, gs in blocks for g in gs]
    v = stat.reindex(allg)[cts].to_numpy().ravel()
    v = v[np.isfinite(v)]
    vmax = max(float(np.nanquantile(np.abs(v), qlim)), 1e-9)
    n_rows = len(allg)
    bar_max = float(np.nanmax(tt.loc[allg, 'share_%'].to_numpy())) * 1.06
    lab_mm = 1.05 * max(len(str(c)) for c in cts) + 4.0
    h_mm = (n_rows + 1.6 * len(blocks)) * row_mm + lab_mm + 14.0
    with plt.rc_context({'font.size': base_pt, 'xtick.labelsize': base_pt, 'ytick.labelsize': base_pt}):
        fig = mm_fig(width_mm, h_mm)
        gs_ = fig.add_gridspec(len(blocks), 3, height_ratios=[len(g) for _, g in blocks],
                               width_ratios=[1, bar_frac, 0.045], hspace=1.6 / max(n_rows, 1), wspace=0.05)
        for bi, (bname, rows) in enumerate(blocks):
            last = bi == len(blocks) - 1
            ax = fig.add_subplot(gs_[bi, 0])
            sns.heatmap(stat.reindex(rows)[cts], ax=ax, cmap=DIVERGING, center=0, vmin=-vmax, vmax=vmax,
                        cbar=False, linewidths=0, xticklabels=last,
                        yticklabels=[f'{g} ({_sign_tag(sign.get(g, 0.0))})' + (' †' if g in GR_CANONICAL else '')
                                     for g in rows])
            ax.grid(False)
            ax.set_xlabel('')
            ax.set_ylabel(bname, labelpad=2)
            ax.tick_params(length=1.2, width=0.4, pad=1.2)
            if last:
                ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
            C_ = contrib.reindex(rows)[cts].to_numpy()
            for yi, xi in zip(*np.where(np.isfinite(C_) & (C_ <= 0))):
                ax.plot(xi + 0.5, yi + 0.5, 'o', ms=base_pt * 0.22, mfc='none', mec='#3d3d3d', mew=0.35)
            axb = fig.add_subplot(gs_[bi, 1])
            axb.barh(np.arange(len(rows)) + 0.5, tt.loc[rows, 'share_%'].to_numpy(), height=0.7,
                     color=bar_color, linewidth=0)
            axb.set_ylim(len(rows), 0)
            axb.set_xlim(0, bar_max)
            axb.set_yticks([])
            axb.tick_params(length=1.2, width=0.4, pad=1.2, labelbottom=last)
            for sp in ('top', 'right', 'left'):
                axb.spines[sp].set_visible(False)
            if last:
                axb.set_xlabel(bar_label, labelpad=2)
        cax = fig.add_subplot(gs_[:, 2])
        fig.colorbar(ScalarMappable(Normalize(-vmax, vmax), DIVERGING), cax=cax, label='DESeq2 Wald statistic')
        cax.tick_params(length=1.5, width=0.4, pad=1.5)
        if title:
            fig.suptitle(title)
    return fig


for c in CONTRASTS:
    for s in DRIVER_TFS:
        tt = tables[(s, c)]
        up, dn = top_drivers(tt)
        fig = plot_target_drivers(le, s, stroma_ct, (up, dn), c, tt, bar_color=RECEPTOR_STYLE[s][0],
                                  title=f'{s} regulon: target genes driving the activity change, {c.replace("_", " ")} (stroma)')
        save_fig(fig, f'{PREFIX}_targetDrivers_{s}_stroma_{c}')
        plt.show()


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_targetDrivers_AR_stroma_paed_vs_infant.pdf  (120 x 112 mm)


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_targetDrivers_NR3C1_stroma_paed_vs_infant.pdf  (120 x 139 mm)


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_targetDrivers_PGR_stroma_paed_vs_infant.pdf  (120 x 119 mm)


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_targetDrivers_AR_stroma_adult_vs_infant.pdf  (120 x 143 mm)


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_targetDrivers_NR3C1_stroma_adult_vs_infant.pdf  (120 x 143 mm)


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_targetDrivers_PGR_stroma_adult_vs_infant.pdf  (120 x 123 mm)


In [25]:
def pool_sources(le, allowed, contrast, name):
    """Sum each gene's per-receptor share (of that receptor's target-side r, per population) over the
    receptors it is a target of. `allowed` = {receptor: populations in which it may be pooled}."""
    d = le[le['contrast'] == contrast]
    keep = pd.concat([d[(d['source'] == s) & d['cell_type'].isin(cts)] for s, cts in allowed.items() if cts],
                     ignore_index=True)
    tot = keep.groupby(['source', 'cell_type'])['contrib'].transform('sum')
    keep = keep.assign(share=100 * keep['contrib'] / tot.where(tot.abs() >= R_FLOOR))
    g = keep.dropna(subset=['share']).groupby(['cell_type', 'gene'])
    out = g.agg(contrib=('share', 'sum'), stat=('stat', 'first'), weight=('weight', 'mean'),
                n_sources=('source', 'nunique')).reset_index()
    return out.assign(source=name, contrast=contrast)


POOLED = 'AR+NR3C1+PGR'
POOLED_RUNS = {}                       # per contrast: (pooled frame, target table, rising, falling)
for c in CONTRASTS:
    # a receptor is pooled only in populations where its activity rises (z > 0)
    allowed = {s: sorted(res.loc[(res['source'] == s) & (res['contrast'] == c) & res['cell_type'].isin(stroma_ct)
                                 & (res['z'] > 0), 'cell_type']) for s in DRIVER_TFS}
    pool = pool_sources(le, allowed, c, POOLED)
    tt_pool = target_table(pool, POOLED, stroma_ct, c, r_floor=1e-9)
    tt_pool['n_receptors'] = pool.groupby('gene')['n_sources'].max().reindex(tt_pool.index)
    tt_pool.to_csv(f'{RESULTS}/{PREFIX}_targetDrivers_pooledSteroid_stroma_{c}.csv')
    up, dn = top_drivers(tt_pool, n=20)
    POOLED_RUNS[c] = (pool, tt_pool, up, dn)
    print(f'{c}: pooled over', {s: len(v) for s, v in allowed.items()}, 'populations per receptor')
    display(tt_pool.loc[up + dn, ['weight', 'share_%', 'mean_stat', 'n_pops', 'n_drives', 'n_receptors',
                                  'canonical_GC_AR_target']].round(2))
    fig = plot_target_drivers(pool, POOLED, stroma_ct, (up, dn), c, tt_pool, bar_color='0.45',
                              bar_label='% of pooled steroid signal',
                              title=f'Pooled AR / NR3C1 / PGR programme: driver genes, {c.replace("_", " ")} (stroma)')
    save_fig(fig, f'{PREFIX}_targetDrivers_pooledSteroid_stroma_{c}')
    plt.show()


paed_vs_infant: pooled over {'AR': 7, 'NR3C1': 7, 'PGR': 8} populations per receptor


,weight,share_%,mean_stat,n_pops,n_drives,n_receptors,canonical_GC_AR_target
gene,,,,,,,
KLF5,1.0,6.41,1.95,8,8,3,False
CEBPD,1.0,6.15,2.97,8,7,2,True
PNRC1,1.0,5.86,2.71,8,7,2,False
EGFR,1.0,5.35,1.69,8,8,3,False
CEBPB,1.0,5.23,2.38,7,7,1,False
FKBP5,1.0,5.14,1.59,8,8,3,True
PER2,1.0,4.99,2.20,7,7,1,False
DUSP1,1.0,4.45,1.79,8,8,2,True
NFKBIA,1.0,4.42,1.45,8,8,2,True


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_targetDrivers_pooledSteroid_stroma_paed_vs_infant.pdf  (120 x 173 mm)
adult_vs_infant: pooled over {'AR': 9, 'NR3C1': 9, 'PGR': 8} populations per receptor


,weight,share_%,mean_stat,n_pops,n_drives,n_receptors,canonical_GC_AR_target
gene,,,,,,,
FKBP5,1.00,6.02,4.49,9,9,3,True
MT2A,1.00,3.71,5.16,9,9,2,True
KLF5,1.00,3.40,2.64,9,9,3,False
SLPI,1.00,3.01,3.52,9,9,2,False
IGFBP5,1.00,2.99,5.78,8,7,1,False
IRS2,1.00,2.73,2.27,9,7,3,True
PLIN2,1.00,2.31,3.39,8,7,1,True
PGR,1.00,2.24,3.41,6,6,1,False
CEBPD,1.00,2.22,2.89,9,9,2,True


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_targetDrivers_pooledSteroid_stroma_adult_vs_infant.pdf  (120 x 177 mm)


### Rising arm only

The same panels restricted to the activated targets that rise with age (the upper block of the
figures above; same genes, same shares), for use where only the induced programme is shown.


In [26]:
for c in CONTRASTS:
    for s in DRIVER_TFS:
        tt = tables[(s, c)]
        up, dn = top_drivers(tt)
        fig = plot_target_drivers(le, s, stroma_ct, (up, dn), c, tt, bar_color=RECEPTOR_STYLE[s][0],
                                  arms=('rising',),
                                  title=f'{s} regulon: activated targets rising with age, {c.replace("_", " ")} (stroma)')
        save_fig(fig, f'{PREFIX}_targetDrivers_{s}_stroma_{c}_risingOnly')
        plt.show()
    pool, tt_pool, up, dn = POOLED_RUNS[c]
    fig = plot_target_drivers(pool, POOLED, stroma_ct, (up, dn), c, tt_pool, bar_color='0.45',
                              bar_label='% of pooled steroid signal', arms=('rising',),
                              title=f'Pooled AR / NR3C1 / PGR programme: targets rising with age, {c.replace("_", " ")} (stroma)')
    save_fig(fig, f'{PREFIX}_targetDrivers_pooledSteroid_stroma_{c}_risingOnly')
    plt.show()


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_targetDrivers_AR_stroma_paed_vs_infant_risingOnly.pdf  (120 x 83 mm)


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_targetDrivers_NR3C1_stroma_paed_vs_infant_risingOnly.pdf  (120 x 83 mm)


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_targetDrivers_PGR_stroma_paed_vs_infant_risingOnly.pdf  (120 x 83 mm)


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_targetDrivers_pooledSteroid_stroma_paed_vs_infant_risingOnly.pdf  (120 x 100 mm)


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_targetDrivers_AR_stroma_adult_vs_infant_risingOnly.pdf  (120 x 87 mm)


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_targetDrivers_NR3C1_stroma_adult_vs_infant_risingOnly.pdf  (120 x 87 mm)


/tmp/ipykernel_1020638/2561703762.py:35: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  return plt.figure(figsize=(width_mm * MM, height_mm * MM), layout='constrained')


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_targetDrivers_PGR_stroma_adult_vs_infant_risingOnly.pdf  (120 x 87 mm)


saved thyAgeing_CollecTRI_ageDE_taa_l2_v2_targetDrivers_pooledSteroid_stroma_adult_vs_infant_risingOnly.pdf  (120 x 104 mm)


## Outputs

All under `results/` and `figures/` with the prefix `thyAgeing_CollecTRI_ageDE_<level>_`:

**Tables**
- `scores_perCellType.csv` — the Supplementary Table: every cell type × contrast × TF with raw score,
  permutation null, `z`, `padj`, percentiles.
- `steroidReceptors_table.csv` — the five receptors only.
- `sharedRanking_stroma.csv`, `top20_sharedStroma.csv` — the consistency ranking across the stroma
  and the two top-20 lists.
- `stressControl_*.csv` — stress-TF ranks, contribution decomposition, drop-and-rescore, split
  regulons, and the summary.
- `steroidReceptors_targetContributions_stroma.csv`, `steroidReceptors_targetDrivers_stroma.csv`,
  `targetDrivers_pooledSteroid_stroma_<contrast>.csv` — per-gene contributions and driver rankings.

**Figures** (PDF, sized in mm for print)
- `steroidReceptors_violins_{stroma,allCellTypes}.pdf`, `steroidReceptors_percentiles_{stroma,allCellTypes}.pdf`
- `top20_sharedStroma.pdf`
- `stressControl_TFranks_stroma.pdf`
- `targetDrivers_<receptor>_stroma_<contrast>.pdf`, `targetDrivers_pooledSteroid_stroma_<contrast>.pdf`, and their `_risingOnly` versions (upper block only)
